# 🏇 KEIBA-AI 金曜予想 ノートブック v5

v5変更点: Gmail削除 / 全券種対応(複勝・単勝・ワイド・馬連・馬単・三連複) / EV×条件スコア選択 / bet_simulation記録

## セル1: セットアップ

In [11]:
!pip install -q requests beautifulsoup4 lxml pandas
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json, math, re, time, sqlite3, smtplib, requests, statistics, pickle
import subprocess
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timezone, timedelta
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
for d in [BASE_DIR, f'{BASE_DIR}/data', f'{BASE_DIR}/logs', f'{BASE_DIR}/app']:
    os.makedirs(d, exist_ok=True)

# ④ Colabの時計ずれ恒久対策
# ntpdateで時刻補正を試み、失敗したらJST固定で計算する
def get_jst_now():
    """常に正確なJST時刻を返す"""
    try:
        result = subprocess.run(
            ['sudo', 'ntpdate', '-u', 'ntp.nict.jp'],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode != 0:
            raise Exception('ntpdate failed')
    except Exception:
        pass  # 失敗してもJST固定で続行
    # UTC取得してJSTに変換（システム時計に依存しない）
    JST = timezone(timedelta(hours=9))
    return datetime.now(JST)

jst_now = get_jst_now()
print(f'✅ セットアップ完了')
print(f'   現在時刻(JST): {jst_now.strftime("%Y-%m-%d %H:%M:%S")}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ セットアップ完了
   現在時刻(JST): 2026-05-16 05:45:25


## セル2: 設定

In [12]:
BANKROLL    = 100_000
TOP_N_RACES = 6

# ---- 賭け金設定（3連単・枠連は対象外）----
FUKU_AMT = 500
TAN_AMT  = 300
WIDE_AMT = 300
REN_AMT  = 300
TAN2_AMT = 300
SAN_AMT  = 300

W = {
    'pace':0.25,'recent':0.20,'jockey':0.15,'trainer':0.10,
    'blood':0.10,'distance':0.08,'post':0.06,'bias':0.04,'weight':0.02,
}
POST_BIAS = {
    '東京':+1.5,'新潟':+1.5,'中京':-1.5,'阪神':-0.5,'福島':-0.5,
    '中山':+0.5,'小倉':+0.5,'札幌':0.0,'函館':0.0,'京都':0.0,
}
VENUE_ORDER = {
    '中山':0,'東京':1,'阪神':2,'京都':3,'中京':4,
    '新潟':5,'福島':6,'小倉':7,'札幌':8,'函館':9
}

_wpath = f'{BASE_DIR}/data/optimal_weights.json'
if os.path.exists(_wpath):
    with open(_wpath) as _f: _opt = json.load(_f)
    _w_keys = ['pace','recent','jockey','trainer','blood','distance','post','bias','weight']
    _new_W = {k: _opt[k] for k in _w_keys if k in _opt}
    if _new_W and abs(sum(_new_W.values())-1.0) < 0.05:
        W = _new_W
        if 'post_bias_by_course' in _opt: POST_BIAS = _opt['post_bias_by_course']
        print(f'✅ チューニング済み重みを読み込み（AUC:{_opt.get("val_auc","?")}）')
    else: print('⚠ optimal_weights不正 → デフォルト重みを使用')
else: print('ℹ チューニング未実施 → デフォルト重みを使用')

print(f'✅ 設定完了 複勝¥{FUKU_AMT} 単勝¥{TAN_AMT} ワイド¥{WIDE_AMT} 馬連¥{REN_AMT} 馬単¥{TAN2_AMT} 三連複¥{SAN_AMT}')

# ── XGBoost複勝予測モデル読み込み（新設計）────────────────
_XGB_FUKUSHO_MODEL = None
_XGB_FEATURE_COLS  = None
_xgb_fpath  = f'{BASE_DIR}/data/xgb_fukusho_model.pkl'
_xgb_fcpath = f'{BASE_DIR}/data/xgb_feature_cols.json'
if os.path.exists(_xgb_fpath) and os.path.exists(_xgb_fcpath):
    import pickle as _pkl3
    with open(_xgb_fpath,'rb') as _f:  _XGB_FUKUSHO_MODEL = _pkl3.load(_f)
    with open(_xgb_fcpath,encoding='utf-8') as _f:
        _finfo = json.load(_f)
        _XGB_FEATURE_COLS = _finfo['feature_cols']
    print(f'✅ XGBoost複勝モデル読み込み完了（{len(_XGB_FEATURE_COLS)}特徴量）')
    print(f'   訓練期間: {_finfo.get("train_period","不明")} / 検証AUC: {_finfo.get("val_auc",0):.4f}')
else:
    print('⚠ XGBoost複勝モデルなし → 重み合算方式で代替（重みチューニングを先に実行してください）')


✅ チューニング済み重みを読み込み（AUC:0.6501413551617732）
✅ 設定完了 複勝¥500 単勝¥300 ワイド¥300 馬連¥300 馬単¥300 三連複¥300
✅ XGBoost複勝モデル読み込み完了（62特徴量）
   訓練期間: 2025-01-05〜2026-03-29 / 検証AUC: 0.6501


## セル3: DB読み込み

In [13]:
def load_dbs():
    paths = {
        'jockey':  f'{BASE_DIR}/data/jockey_db.csv',
        'trainer': f'{BASE_DIR}/data/trainer_db.csv',
        'style':   f'{BASE_DIR}/data/horse_style_db.csv',
    }
    jd = pd.read_csv(paths['jockey'])  if os.path.exists(paths['jockey'])  else pd.DataFrame()
    td = pd.read_csv(paths['trainer']) if os.path.exists(paths['trainer']) else pd.DataFrame()
    sd = pd.read_csv(paths['style'])   if os.path.exists(paths['style'])   else pd.DataFrame()
    jdict = {(r['騎手'],r['競馬場'],r['surface']): r['勝率'] for _,r in jd.iterrows()} if len(jd) else {}
    tdict = dict(zip(td['調教師'], td['勝率'])) if len(td) else {}
    sdict = dict(zip(sd['馬名'], sd['安定脚質'])) if len(sd) else {}
    print(f'✅ 実績DB読み込み完了')
    print(f'  騎手DB: {len(jdict)}件 / 調教師DB: {len(tdict)}件 / 脚質DB: {len(sdict)}頭')
    return jdict, tdict, sdict

jdict, tdict, sdict = load_dbs()

# 距離・競馬場適性DB
horse_dist_dict   = {}
horse_course_dict = {}
for fn, d in [('horse_dist_dict.pkl', horse_dist_dict),
              ('horse_course_dict.pkl', horse_course_dict)]:
    p = f'{BASE_DIR}/data/{fn}'
    if os.path.exists(p):
        with open(p,'rb') as f:
            d.update(pickle.load(f))
print(f'  距離適性DB: {len(horse_dist_dict):,}件 / 競馬場適性DB: {len(horse_course_dict):,}件')

# バイアス読み込み
bias = None
bf = f'{BASE_DIR}/data/track_bias_latest.json'
if os.path.exists(bf):
    with open(bf) as f: bias = json.load(f)
    print(f'  バイアス({bias.get("date","")}): {bias.get("summary","フラット")}')
else:
    print('  バイアス: なし')


✅ 実績DB読み込み完了
  騎手DB: 930件 / 調教師DB: 192件 / 脚質DB: 9528頭
  距離適性DB: 4,941件 / 競馬場適性DB: 2,518件
  バイアス(2026-05-16): フラット


## セル3b: src/ モジュールセットアップ（初回のみ）


In [ ]:
import urllib.request, os, sys
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
files = [
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/features/engine.py',
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    'src/scraper/calendar.py',
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/predict.py',
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}', dest)
    print(f'✅ {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
print('🔄 完了')


## セル4: スクレイパー定義

In [14]:
import sys
sys.path.insert(0, f'{BASE_DIR}/src')
try:
    from utils.db import init_db, save_bets_db, save_results_db
    from utils.jst import get_jst_now
    from scraper.jra_scraper import fetch_races_on_date, fetch_results
    print("✅ src/ モジュール読み込み成功")
except ImportError as e:
    print(f"⚠ src/ なし → ノートブック内定義を使用 ({e})")

HEADERS     = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
JRA_BASE    = 'https://www.jra.go.jp'
PLACE_NAMES = {'01':'札幌','02':'函館','03':'福島','04':'新潟','05':'東京',
               '06':'中山','07':'中京','08':'京都','09':'阪神','10':'小倉'}
PLACE_ENG   = {'nakayama':'06','hanshin':'09','chukyo':'07','tokyo':'05',
               'kyoto':'08','sapporo':'01','hakodate':'02','fukushima':'03',
               'niigata':'04','kokura':'10'}

# ★ 開催カレンダー（年間開催日割PDFより）
KAISAI_CALENDAR = {
    "06": [  # 中山
        {"kai":"03","days":["20260328","20260329","20260404","20260405",
                            "20260411","20260412","20260418","20260419"]}],
    "05": [  # 東京
        {"kai":"02","days":["20260425","20260426",
                            "20260502","20260503","20260509","20260510",
                            "20260516","20260517","20260523","20260524",
                            "20260530","20260531","20260606","20260607"]}],
    "09": [  # 阪神
        {"kai":"02","days":["20260328","20260329","20260404","20260405",
                            "20260411","20260412","20260418","20260419"]},
        {"kai":"03","days":["20260510","20260516","20260517",
                            "20260523","20260524","20260530","20260531",
                            "20260606","20260607","20260613","20260614",
                            "20260620","20260621","20260627","20260628"]}],
    "08": [  # 京都
        {"kai":"03","days":["20260425","20260426",
                            "20260502","20260503","20260509","20260510",
                            "20260516","20260517","20260523","20260524",
                            "20260530","20260531","20260606","20260607"]}],
    "03": [  # 福島
        {"kai":"01","days":["20260411","20260412","20260418","20260419","20260425","20260426"]}],
    "04": [  # 新潟
        {"kai":"01","days":["20260502","20260503","20260509","20260510","20260516","20260517"]}],
    "02": [  # 函館
        {"kai":"01","days":["20260606","20260607","20260613","20260614",
                            "20260620","20260621","20260627","20260628"]}],
    "07":[], "10":[], "01":[],
}

def get_base_from_calendar(place_code, date_str):
    for entry in KAISAI_CALENDAR.get(place_code, []):
        if date_str in entry['days']:
            nichi = entry['days'].index(date_str) + 1
            return f"pw01dde01{place_code}{date_str[:4]}{entry['kai']}{nichi:02d}"
    return None

def get_kaisai_on_date(date_str, sess):
    links = {}
    for pc in KAISAI_CALENDAR:
        base = get_base_from_calendar(pc, date_str)
        if base:
            links[base] = date_str
            print(f"  📅 {PLACE_NAMES.get(pc,'?')} → {base}")
    # JRAサイトのthisweekページで当日開催会場を確認
    # → thisweekに載っている会場のみを有効とする（KAISAI_CALENDARの誤り補正）
    thisweek_pcs = set()
    try:
        resp = sess.get(f'{JRA_BASE}/keiba/thisweek/', timeout=15)
        resp.encoding = 'shift_jis'
        for a in BeautifulSoup(resp.text,'lxml').find_all('a', href=True):
            href = a['href']
            if 'pw01dde01' not in href: continue
            m = re.search(r'pw01dde01(\d{2})(\d{4})(\d{2})(\d{2})(\d{8})', href)
            if not m: continue
            pc_tw = m.group(1)
            date_tw = m.group(5)
            if date_tw != date_str: continue
            thisweek_pcs.add(pc_tw)
            base_tw = f'pw01dde01{m.group(1)}{m.group(2)}{m.group(3)}{m.group(4)}'
            if base_tw not in links:
                links[base_tw] = date_str
                print(f"  🌐 thisweek補完: {PLACE_NAMES.get(pc_tw,'?')} → {base_tw}")
        # thisweekに載っていない会場をフィルタ（非開催会場の除外）
        if thisweek_pcs:
            removed = []
            links = {b: d for b, d in links.items()
                     if re.search(r'pw01dde01(\d{2})', b) and
                        re.search(r'pw01dde01(\d{2})', b).group(1) in thisweek_pcs}
            # 除外された会場を表示
            for pc in KAISAI_CALENDAR:
                if pc not in thisweek_pcs:
                    base_cal = get_base_from_calendar(pc, date_str)
                    if base_cal:
                        print(f"  ⏭ {PLACE_NAMES.get(pc,'?')} カレンダーにあるがthisweekになし → スキップ")
    except Exception as e:
        print(f"  ⚠ thisweek取得失敗: {e}")
    if not links:
        try:
            print(f"  ⚠ {date_str}はカレンダー・thisweek両方になし → 結果一覧から取得")
            resp2 = sess.post(f'{JRA_BASE}/JRADB/accessS.html',
                              data={'CNAME':'pw01sli00/AF'}, timeout=15)
            resp2.encoding = 'shift_jis'
            for tag in BeautifulSoup(resp2.text,'lxml').find_all(onclick=True):
                oc = tag['onclick']
                m = re.search(r'pw01srl10(\d{2})(\d{4})(\d{2})(\d{2})(\d{8})/(\w{2})', oc)
                if m and m.group(5)==date_str:
                    pc_r = m.group(1)
                    base_r = f'pw01dde01{pc_r}{m.group(2)}{m.group(3)}{m.group(4)}'
                    if base_r not in links:
                        links[base_r] = date_str
                        print(f"  📋 結果一覧補完: {PLACE_NAMES.get(pc_r,'?')} → {base_r}")
        except Exception as e:
            print(f"  ⚠ 結果一覧取得失敗: {e}")
    if not links:
        print(f"  ❌ {date_str}の開催情報が見つかりません")
    return links

def calc_suffix(r01, r):
    if r<=9:    return f'{(r01+(r-1)*181)%256:02X}'
    elif r==10: return f'{(r01+8*181+245)%256:02X}'
    else:       return f'{(r01+8*181+245+(r-10)*181)%256:02X}'

def find_r01_shutuba(base, date, sess):
    for s in range(256):
        cn = f'{base}01{date}/{s:02X}'
        r  = sess.post(f'{JRA_BASE}/JRADB/accessD.html',
                       data={'cname':cn,'CNAME':cn}, timeout=10)
        r.encoding = 'shift_jis'
        if 'パラメータエラー' not in r.text and BeautifulSoup(r.text,'lxml').find_all('table'):
            return s
        time.sleep(0.05)
    return None

def parse_header(text):
    info = {}
    m = re.search(r'(\d{4})年(\d{1,2})月(\d{1,2})日', text)
    if m: info['date'] = f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    for code, name in PLACE_NAMES.items():
        if name in text: info['racecourse'] = name; break
    # ── 障害レース検出（距離・芝ダの正規表現優先、失敗時のみ障害判定）──
    dm = re.search(r'([\d]+)\s*メートル\s*[（(]\s*([芝ダ])[^）)]*([右左])', text)
    if not dm:
        # 芝/ダート距離が見つからない → 先頭1000文字のみで障害確認
        _header = text[:1000]
        if any(kw in _header for kw in ['障害', 'ジャンプ', 'J・G', 'J-G']):
            info['surface'] = '障害'; info['distance'] = 0; info['direction'] = ''
            return info
    # ─────────────────────────────────────────────────────
    if dm:
        info['distance']  = int(dm.group(1).replace(',',''))
        info['surface']   = '芝' if dm.group(2)=='芝' else 'ダート'
        info['direction'] = dm.group(3)
    else:
        info['distance'] = 2000; info['surface'] = '芝'; info['direction'] = '右'
    for kw, cls in [('G1','G1'),('G2','G2'),('G3','G3'),
                    ('3勝クラス','3勝クラス'),('2勝クラス','2勝クラス'),
                    ('1勝クラス','1勝クラス'),('未勝利','未勝利'),('新馬','新馬'),
                    ('オープン','オープン'),('重賞','重賞')]:
        if kw in text:
            info['class'] = cls
            break
    else:
        info['class'] = '1勝クラス'
    return info

def parse_rname(text, rn):
    c = text.replace('本賞金','').replace('付加賞','')
    sp = re.search(r'([\u3040-\u9fff\u30A0-\u30FFa-zA-Z0-9]+(?:賞|杯|記念|特別|ステークス|カップ|トロフィー))', c)
    if sp:
        n = sp.group(1).strip()
        if n not in ('本賞','付加賞') and len(n)>=3: return n
    gen = re.search(r'(\d歳(?:以上)?(?:未勝利|1勝クラス|2勝クラス|3勝クラス|オープン))', text)
    return gen.group(1).strip() if gen else f'R{rn:02d}'

# ─── クラス判定（parse_histで使用）────────────────────────
def get_history_from_db(horse_name, hist_db_path, limit=5):
    """
    history.dbから馬の直近N走を取得
    完全一致→前方5文字ファジーマッチの順で検索
    """
    try:
        import sqlite3 as _sq
        conn = _sq.connect(hist_db_path)

        # まず完全一致で検索
        rows = conn.execute('''
            SELECT h.race_id, h.date, h.race_name, h.distance, h.surface,
                   h.place, h.agari3f, h.popularity,
                   h.corner_3, r.first_3f, h.horse_num
            FROM horse_history h
            LEFT JOIN race_history r ON h.race_id = r.race_id
            WHERE h.horse_name = ?
            ORDER BY h.date DESC, h.race_id DESC
            LIMIT ?
        ''', (horse_name, limit)).fetchall()

        # 完全一致でヒットしなければ前方5文字で再検索
        if not rows and len(horse_name) >= 4:
            rows = conn.execute('''
                SELECT h.race_id, h.date, h.race_name, h.distance, h.surface,
                       h.place, h.agari3f, h.popularity,
                       h.corner_3, r.first_3f, h.horse_num
                FROM horse_history h
                LEFT JOIN race_history r ON h.race_id = r.race_id
                WHERE h.horse_name LIKE ?
                ORDER BY h.date DESC, h.race_id DESC
                LIMIT ?
            ''', (horse_name[:5] + '%', limit)).fetchall()

        if not rows:
            conn.close()
            return []

        results = []
        for row in rows:
            race_id, date, race_name, distance, surface, place, agari3f, popularity, corner_3, first_3f_val, horse_num_val = row
            finishers = conn.execute(
                'SELECT COUNT(*) FROM horse_history WHERE race_id=?', (race_id,)
            ).fetchone()[0]
            winner = conn.execute(
                'SELECT agari3f FROM horse_history WHERE race_id=? AND place=1',
                (race_id,)
            ).fetchone()
            if winner and winner[0] and agari3f and place > 1:
                margin = max(0.0, round((agari3f - winner[0]) * 0.3, 2))
            else:
                margin = 0.0
            if agari3f:
                all_agari = conn.execute(
                    'SELECT agari3f FROM horse_history WHERE race_id=? AND agari3f IS NOT NULL',
                    (race_id,)
                ).fetchall()
                all_vals = sorted([x[0] for x in all_agari])
                if all_vals:
                    rank = sum(1 for v in all_vals if v < agari3f)
                    agari3f_rank_pct = rank / max(len(all_vals)-1, 1)
                else:
                    agari3f_rank_pct = 0.5
            else:
                agari3f_rank_pct = 0.5
            results.append({
                'place':            place,
                'finishers':        max(finishers, 1),
                'distance':         distance,
                'surface':          surface,
                'class':            None if re.match(r'^R\d{2}$', race_name) else get_class_from_racename(race_name),
                'margin':           margin,
                'agari3f_rank_pct': round(agari3f_rank_pct, 3),
                'condition':        '良',
                'date':             date,
                'last_3f':          agari3f,
                'first_3f':         first_3f_val,
                'corner_3':         corner_3,
                'race_id':          race_id,
            })
        conn.close()
        return results
    except Exception:
        return []

def get_class_from_racename(rname: str) -> str:
    """レース名からクラスを確実に判定"""
    if not rname: return '1勝クラス'
    if 'G1' in rname or '（G1）' in rname: return 'G1'
    if 'G2' in rname or '（G2）' in rname: return 'G2'
    if 'G3' in rname or '（G3）' in rname: return 'G3'
    if any(kw in rname for kw in ['ステークス','記念','特別','カップ','賞','杯','トロフィー']):
        return 'オープン'
    if '3勝クラス' in rname or '3勝' in rname: return '3勝クラス'
    if '2勝クラス' in rname or '2勝' in rname: return '2勝クラス'
    if '1勝クラス' in rname or '1勝' in rname: return '1勝クラス'
    if '未勝利' in rname: return '未勝利'
    if '新馬' in rname: return '新馬'
    return '1勝クラス'

def parse_hist(text):
    """出走表の過去走履歴をパース"""
    if not text or len(text)<10: return None
    h = {}
    pm = re.search(r'(\d+)\s*着', text); h['place'] = int(pm.group(1)) if pm else 10
    fm = re.search(r'(\d+)\s*頭', text); h['finishers'] = int(fm.group(1)) if fm else 16
    dm = re.search(r'(\d{4})(?:芝|ダ)', text)
    if not dm: dm = re.search(r'(\d{4})', text)
    h['distance'] = int(dm.group(1)) if dm else 2000
    h['surface']  = 'ダート' if 'ダ' in text else '芝'
    for cond in ['不良','重','稍重','良']:
        if cond in text: h['condition'] = cond; break
    else: h['condition'] = '良'
    h['agari3f_rank_pct'] = 0.5
    margin = 0.0
    mm = re.search(r'(\d+\.\d+)秒', text)
    if mm: margin = float(mm.group(1))
    elif 'クビ' in text: margin = 0.1
    elif 'ハナ' in text: margin = 0.05
    elif 'アタマ' in text: margin = 0.07
    h['margin'] = margin
    # レース名からクラスを判定（確実な方法）
    h['class'] = get_class_from_racename(text)
    return h

def parse_horse(cells, rc, surf):
    """馬名スクレイピングバグ修正版"""
    if len(cells)<4: return None
    try:
        tx = [c.get_text(' ',strip=True) for c in cells]
        umaban = None
        for col_idx in [0,1,2]:
            if col_idx>=len(tx): break
            nm = re.match(r'^\s*(\d{1,2})\s*$', tx[col_idx])
            if nm: umaban=int(nm.group(1)); break
        if umaban is None: return None
        name = ''
        for col_idx in [2,3]:
            if col_idx>=len(tx): continue
            name_m = re.match(
                r'^([\u30A0-\u30FF\u4e00-\u9fffA-Za-z][\u30A0-\u30FF\u4e00-\u9fffA-Za-z0-9・]{1,20})',
                tx[col_idx].strip())
            if name_m: name=name_m.group(1).strip(); break
        if not name or len(name)<2: return None
        odds_m=pop_m=None
        for col_idx in [2,3,4]:
            if col_idx>=len(tx): continue
            if not odds_m: odds_m=re.search(r'(\d+\.\d+)\s*\(\d+\s*番人気',tx[col_idx])
            if not pop_m:  pop_m =re.search(r'\((\d+)\s*番人気',tx[col_idx])
        win_odds   = float(odds_m.group(1)) if odds_m else None
        popularity = int(pop_m.group(1))    if pop_m  else 99
        col4=tx[4] if len(tx)>4 else ''
        sire_m  = re.search(r'父[：:]([^\s母]{2,20})',col4)
        dams_m  = re.search(r'母の父[：:]([^)）\s]{2,20})',col4)
        train_m = re.search(r'([\u4e00-\u9fff]{2,5}[\u4e00-\u9fff\s]{0,4})\s*[（(][美栗]',col4)
        sire     = sire_m.group(1).strip()  if sire_m  else ''
        dam_sire = dams_m.group(1).strip()  if dams_m  else ''
        trainer  = train_m.group(1).strip() if train_m else ''
        col3=tx[3] if len(tx)>3 else ''
        sa_m=re.search(r'([牡牝セ])(\d+)',col3)
        sex=sa_m.group(1) if sa_m else '牡'; age=int(sa_m.group(2)) if sa_m else 4
        wl_m=re.search(r'(\d{2}\.?\d?)',tx[6] if len(tx)>6 else '')
        weight_load=float(wl_m.group(1)) if wl_m else 56.0
        col7=tx[7] if len(tx)>7 else ''
        jockey=col7.split()[0] if col7.split() else ''
        # DB補完優先：history.dbから正確なデータを取得
        hist_db = f'{BASE_DIR}/data/history.db'
        history = get_history_from_db(name, hist_db, limit=5)
        # DBにない場合は出走表テキストからフォールバック
        if not history:
            history = [h for h in [parse_hist(tx[i]) for i in range(8,min(12,len(tx)))] if h]
        running_style=sdict.get(name)
        if not running_style:
            avg_a=sum(h.get('agari3f_rank_pct',.5) for h in history)/len(history) if history else .5
            avg_p=sum(h.get('place',5) for h in history)/len(history) if history else 5
            running_style=('差し' if avg_a<0.25 and avg_p<=3 else
                           '追込' if avg_a<0.15 else '先行' if avg_p<=2.5 else '差し')
        good=[h['distance'] for h in history if h.get('place',10)<=3]
        best_dist=int(sum(good)/len(good)) if good else (history[0]['distance'] if history else 2000)
        return {
            'num':umaban,'name':name,'age':age,'sex':sex,
            'sire':sire,'dam_sire':dam_sire,'jockey':jockey,'trainer':trainer,
            'weight_load':weight_load,'popularity':popularity,'post_position':umaban,
            'running_style':running_style,'win_odds':win_odds,
            'history':history,'best_distance':best_dist,
            'prev_distance':history[0].get('distance',2000) if history else 2000,
            'jockey_rate':jdict.get((jockey,rc,surf),0.15),
            'trainer_rate':tdict.get(trainer,0.12),
        }
    except: return None

def parse_soup(soup, cname, rc, date, rn):
    try:
        tables = soup.find_all('table')
        if not tables: return None
        header = tables[0].get_text(' ', strip=True)
        info   = parse_header(header); info['racecourse'] = rc
        rname  = parse_rname(header, rn)
        if info.get('surface') == '障害' or any(kw in rname for kw in ['ジャンプ','障害','(J)','（J）','J・G']): return None
        surf   = info.get('surface','芝')
        horses = [h for h in [
            parse_horse(row.find_all('td'), rc, surf)
            for row in tables[0].find_all('tr') if len(row.find_all('td'))>=4
        ] if h]
        if not horses: return None
        for h in horses:
            if h['win_odds'] is None: h['win_odds'] = round(h['popularity']*2.5, 1)
        m = re.search(r'pw01dde01(\d{2})(\d{4})(\d{2})(\d{2})(\d{2})(\d{8})', cname)
        race_id = f"{m.group(6)}_{m.group(1)}_{rn:02d}" if m else cname[-12:]
        esc   = sum(1 for h in horses if h.get('running_style') == '逃げ')
        front = sum(1 for h in horses if h.get('running_style') == '先行')
        clos  = sum(1 for h in horses if h.get('running_style') in ('差し','追込'))
        n     = max(len(horses), 1)
        pace_score = (esc*0.5 + front*0.2 - clos*0.1) / n
        if   pace_score >= 0.15: expected_pace = 'ハイペース'
        elif pace_score <= 0.05: expected_pace = 'スローペース'
        else:                    expected_pace = '平均'
        race_class = info.get('class', get_class_from_racename(rname))
        return {
            'id':            race_id,
            'url':           cname,
            'date':          f'{date[:4]}-{date[4:6]}-{date[6:]}',
            'racecourse':    rc,
            'race_name':     rname,
            'race_num':      rn,
            'class':         race_class,
            'distance':      info['distance'],
            'surface':       surf,
            'direction':     info.get('direction','右'),
            'condition':     '良',
            'num_horses':    len(horses),
            'expected_pace': expected_pace,
            'pace_score':    round(pace_score, 4),
            'escape_count':  esc,
            'front_count':   front,
            'closing_count': clos,
            'horses':        horses,
        }
    except:
        return None

def fetch_races_on_date(date_str, sess):
    """カレンダーから一発でbaseを生成してレース取得"""
    print(f'📡 {date_str} レース取得中...')
    links = get_kaisai_on_date(date_str, sess)
    if not links: print('  ⚠ 開催情報なし'); return []
    races = []
    for base, date in links.items():
        pc = base[9:11]; rc = PLACE_NAMES.get(pc,'?')
        print(f'\n🏟 {rc}  suffix探索...', end=' ', flush=True)
        r01 = find_r01_shutuba(base, date, sess)
        if r01 is None: print('❌'); continue
        print(f'✅ {r01:02X}')
        for r in range(1,13):
            sx   = calc_suffix(r01,r); cn = f'{base}{r:02d}{date}/{sx}'
            resp = sess.post(f'{JRA_BASE}/JRADB/accessD.html',
                             data={'cname':cn,'CNAME':cn}, timeout=15)
            resp.encoding = 'shift_jis'
            soup = BeautifulSoup(resp.text,'lxml')
            if not soup.find_all('table'): continue
            race = parse_soup(soup,cn,rc,date,r)
            if not race: continue
            races.append(race)
            print(f'  R{r:02d}: {race["race_name"]} {race["distance"]}m {race["surface"]} {race["num_horses"]}頭')
            time.sleep(0.8)
    print(f'\n📋 取得完了: {len(races)}レース')
    return races

print('✅ スクレイパー定義完了（get_class_from_racename追加）')

✅ スクレイパー定義完了（get_class_from_racename追加）


## セル4b: 結果取得・バイアス・照合 定義

In [15]:
def calc_suffix(r01, r):
    if r<=9:    return f'{(r01+(r-1)*181)%256:02X}'
    elif r==10: return f'{(r01+8*181+245)%256:02X}'
    else:       return f'{(r01+8*181+245+(r-10)*181)%256:02X}'

def find_r01_result(base, date, sess):
    for s in range(256):
        cn = f'{base}01{date}/{s:02X}'
        r  = sess.post(f'{JRA_BASE}/JRADB/accessS.html',data={'CNAME':cn},timeout=10)
        r.encoding = 'shift_jis'
        if 'パラメータエラー' not in r.text and BeautifulSoup(r.text,'lxml').find_all('table'):
            return s
        time.sleep(0.05)
    return None

def parse_dividends(soup):
    text=soup.get_text(' ',strip=True); divs={}
    m=re.search(r'単勝\s+(\d+)\s+([\d,]+)\s*円',text)
    if m: divs['tansho']={'num':int(m.group(1)),'payout':int(m.group(2).replace(',',''))}
    idx=text.find('複勝')
    if idx>=0:
        fm=re.findall(r'(\d+)\s+([\d,]+)\s*円',text[idx:idx+200])
        if fm: divs['fukusho']=[{'num':int(f[0]),'payout':int(f[1].replace(',',''))} for f in fm[:3]]
    idx=text.find('ワイド')
    if idx>=0:
        wm=re.findall(r'(\d+)-(\d+)\s+([\d,]+)\s*円',text[idx:idx+300])
        if wm: divs['wide']=[{'nums':[int(w[0]),int(w[1])],'payout':int(w[2].replace(',',''))} for w in wm[:3]]
    return divs

def parse_result_soup(soup, racecourse, race_num, date, place_code):
    try:
        tables=soup.find_all('table')
        header=tables[0].get_text(' ',strip=True)
        info={'racecourse':racecourse,'race_num':race_num,'race_id':f'{date}_{place_code}_{race_num:02d}'}
        dm=re.search(r'([\d,]+)\s*メートル\s*[（(]\s*([芝ダ])',header)
        info['distance']=int(dm.group(1).replace(',','')) if dm else 2000
        info['surface']='芝' if dm and dm.group(2)=='芝' else 'ダート'
        c=header.replace('本賞金','').replace('付加賞','')
        sp=re.search(r'([぀-鿿゠-ヿa-zA-Z0-9]+(?:賞|杯|記念|特別|ステークス|カップ|トロフィー))',c)
        gen=re.search(r'(\d歳(?:以上)?(?:未勝利|1勝クラス|2勝クラス|3勝クラス|オープン))',header)
        info['race_name']=(sp.group(1).strip() if sp and sp.group(1) not in ('本賞','付加賞') and len(sp.group(1))>=3
                          else gen.group(1).strip() if gen else '')
        finishers=[]
        for row in tables[0].find_all('tr'):
            cells=row.find_all('td')
            if len(cells)<10: continue
            texts=[c.get_text(' ',strip=True) for c in cells]
            pm=re.match(r'^(\d+)$',texts[0].strip())
            if not pm: continue
            place=int(pm.group(1))
            # 全着順を保存（制限なし）
            num_m=re.match(r'^(\d+)$',texts[2].strip())
            num=int(num_m.group(1)) if num_m else 0
            # ② 馬名取得修正
            name_m=re.match(
                r'^([゠-ヿ一-鿿A-Za-z][゠-ヿ一-鿿A-Za-z0-9・]{1,20})',
                texts[3].strip())
            name=name_m.group(1).strip() if name_m else texts[3].strip()[:10]
            pos_nums=re.findall(r'\d+',texts[9] if len(texts)>9 else '')
            if pos_nums:
                positions=[int(n) for n in pos_nums[:4]]; first=positions[0]; avg=sum(positions)/len(positions)
                style='逃げ' if first==1 else '先行' if avg<=3 else '差し' if avg<=7 else '追込'
            else: style='差し'
            agari_m=re.search(r'(\d{2}\.\d)',texts[10]) if len(texts)>10 else None
            agari=float(agari_m.group(1)) if agari_m else 0.0
            pop_m=re.match(r'^(\d+)$',texts[13].strip()) if len(texts)>13 else None
            # 騎手(td[6])・調教師(td[12])
            jockey  = texts[6].strip()  if len(texts)>6  else ''
            trainer = texts[12].strip() if len(texts)>12 else ''
            finishers.append({
                'place':place,'num':num,'name':name,
                'running_style':style,'post_position':num,'agari3f':agari,
                'popularity':int(pop_m.group(1)) if pop_m else 99,
                'jockey':jockey,'trainer':trainer,
                'distance':info['distance'],'surface':info['surface'],
            })
        divs=parse_dividends(soup)
        if not finishers: return None
        info['finishers']=finishers; info['dividends']=divs
        return info
    except: return None

# ── カレンダーベースのfetch_results ──────────────────────
def fetch_results(sess, target_date):
    print(f'📡 {target_date} 結果取得中...')
    all_results=[]
    for pc in KAISAI_CALENDAR:
        base_shutuba=get_base_from_calendar(pc,target_date)
        if not base_shutuba: continue
        base_result=base_shutuba.replace('pw01dde01','pw01sde10')
        rc=PLACE_NAMES.get(pc,'?')
        print(f'\n🏟 {rc}  suffix探索...', end=' ', flush=True)
        r01=find_r01_result(base_result,target_date,sess)
        if r01 is None: print('❌'); continue
        print(f'✅ {r01:02X}')
        for r in range(1,13):
            sx=calc_suffix(r01,r); cn=f'{base_result}{r:02d}{target_date}/{sx}'
            resp=sess.post(f'{JRA_BASE}/JRADB/accessS.html',data={'CNAME':cn},headers=HEADERS,timeout=15)
            resp.encoding='shift_jis'
            if 'パラメータエラー' in resp.text: continue
            soup=BeautifulSoup(resp.text,'lxml')
            if not soup.find_all('table'): continue
            result=parse_result_soup(soup,rc,r,target_date,pc)
            if not result: continue
            all_results.append(result)
            top3=result['finishers'][:3]
            t3=' '.join(f"{h['place']}着#{h['num']}{h['name'][:4]}({h['running_style']})" for h in top3)
            print(f'  R{r:02d}: {result.get("race_name","")} {t3}')
            time.sleep(0.8)
    print(f'\n📋 結果取得完了: {len(all_results)}レース')
    return all_results

# ── analyze_bias（修正版） ────────────────────────────────
AGARI_BASE = {
    ('芝','sp'):34.2,('芝','mi'):34.6,('芝','md'):35.0,('芝','lo'):35.5,
    ('ダート','sp'):37.0,('ダート','mi'):37.5,('ダート','md'):38.0,('ダート','lo'):38.5,
}
def dist_zone_bias(d):
    d=int(d)
    if d<=1400: return 'sp'
    if d<=1800: return 'mi'
    if d<=2200: return 'md'
    return 'lo'

def analyze_bias(results):
    bias_by_course={}
    for rc in set(r['racecourse'] for r in results):
        rc_res=[r for r in results if r['racecourse']==rc]
        io_scores=[]
        for r in rc_res:
            fin=r['finishers']
            if len(fin)<3: continue
            num_h=max(h['post_position'] for h in fin)
            avg_all=(num_h+1)/2
            avg_top3=statistics.mean([h['post_position'] for h in fin[:3]])
            io_scores.append((avg_all-avg_top3)/max(num_h/4,1))
        inner_outer=max(-3,min(3,statistics.mean(io_scores)*2)) if io_scores else 0
        style_cnt=defaultdict(int); total=0
        for r in rc_res:
            for h in r['finishers'][:3]:
                style_cnt[h['running_style']]+=1; total+=1
        front=(style_cnt['逃げ']+style_cnt['先行'])/max(total,1)
        pace_bias=max(-3,min(3,(front-0.45)*6))
        speed_devs=[]
        for r in rc_res:
            fin=r['finishers']
            if not fin: continue
            winner=fin[0]; agari=winner.get('agari3f',0)
            if agari<30: continue
            dist=winner.get('distance',r.get('distance',2000))
            surf=winner.get('surface',r.get('surface','芝'))
            zone=dist_zone_bias(dist)
            base_val=AGARI_BASE.get((surf,zone),35.0)
            speed_devs.append(max(-2,min(2,(base_val-agari)/0.8)))
        track_speed=round(statistics.mean(speed_devs),2) if speed_devs else 0
        parts=[]
        if abs(inner_outer)>=1.0: parts.append('内有利' if inner_outer>0 else '外有利')
        if abs(pace_bias)>=1.0:   parts.append('先行有利' if pace_bias>0 else '差し・追込有利')
        if abs(track_speed)>=0.5: parts.append('時計速め' if track_speed>0 else '時計遅め')
        bias_by_course[rc]={
            'inner_outer':round(inner_outer,2),'pace_bias':round(pace_bias,2),
            'track_speed':round(track_speed,2),'summary':'・'.join(parts) if parts else 'フラット',
            'style_dist':dict(style_cnt),'race_count':len(rc_res),
        }
    return bias_by_course

# ── check_predictions（ワイド対応） ──────────────────────
def check_predictions(all_results):
    conn=sqlite3.connect(f'{BASE_DIR}/data/keiba.db')
    try:
        bets_df=pd.read_sql('SELECT * FROM bets WHERE is_hit=-1',conn)
    except: bets_df=pd.DataFrame()
    conn.close()
    if len(bets_df)==0: return {'hit':0,'total':0,'invested':0,'recovered':0,'roi':0,'details':[]}
    hit=0; total=0; invested=0; recovered=0; details=[]
    conn=sqlite3.connect(f'{BASE_DIR}/data/keiba.db')
    for _,bet in bets_df.iterrows():
        parts=str(bet['race_id']).split('_')
        if len(parts)<3: continue
        date_str,pc,rn_str=parts[0],parts[1],parts[2]
        race_num=int(rn_str); rc=PLACE_NAMES.get(pc,'')
        result=next((r for r in all_results if r.get('racecourse')==rc and r.get('race_num')==race_num),None)
        if not result: continue
        top3=[h['num'] for h in result['finishers'][:3]]
        top1=top3[0] if top3 else 0
        divs=result.get('dividends',{}); total+=1; invested+=int(bet['amount'])
        is_hit=False; payout=0
        if bet['bet_type']=='複勝':
            if int(bet['horse_num']) in top3:
                is_hit=True
                for f in divs.get('fukusho',[]):
                    if f['num']==int(bet['horse_num']): payout=int(bet['amount']*f['payout']/100); break
        elif bet['bet_type']=='単勝':
            if int(bet['horse_num'])==top1:
                is_hit=True; payout=int(bet['amount']*divs.get('tansho',{}).get('payout',0)/100)
        elif bet['bet_type']=='ワイド':
            h1=int(bet['horse_num']); h2=int(bet.get('horse_num2',0))
            for w in divs.get('wide',[]):
                if h1 in w['nums'] and h2 in w['nums']:
                    is_hit=True; payout=int(bet['amount']*w['payout']/100); break
        recovered+=payout
        if is_hit: hit+=1
        conn.execute('UPDATE bets SET is_hit=?,payout=? WHERE id=?',(1 if is_hit else 0,payout,bet['id']))
        mark='✅' if is_hit else '❌'
        suffix=f'→¥{payout:,}' if is_hit else '→外れ'
        details.append(f'  {mark} {rc}R{race_num:02d} {result.get("race_name","")[:6]} {bet["bet_type"]}#{int(bet["horse_num"])} ¥{int(bet["amount"]):,}{suffix}')
    conn.commit(); conn.close()
    roi=recovered/invested*100 if invested>0 else 0
    return {'hit':hit,'total':total,'invested':invested,'recovered':recovered,'roi':roi,'details':details}

def save_results_db(all_results):
    conn=sqlite3.connect(f'{BASE_DIR}/data/keiba.db')
    conn.execute('''CREATE TABLE IF NOT EXISTS results (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        race_id TEXT, place INTEGER, horse_num INTEGER,
        horse_name TEXT, running_style TEXT, jockey TEXT, trainer TEXT,
        agari3f REAL, tansho_payout INTEGER, fukusho_payout INTEGER)''')
    for r in all_results:
        tp=r['dividends'].get('tansho',{}).get('payout',0)
        for h in r['finishers'][:6]:
            fp=next((f['payout'] for f in r['dividends'].get('fukusho',[]) if f['num']==h['num']),0)
            conn.execute(
                'INSERT INTO results (race_id,place,horse_num,horse_name,running_style,jockey,trainer,agari3f,tansho_payout,fukusho_payout) VALUES (?,?,?,?,?,?,?,?,?,?)',
                (r['race_id'],h['place'],h['num'],h['name'],h['running_style'],
                 h.get('jockey',''),h.get('trainer',''),h['agari3f'],
                 tp if h['place']==1 else 0,fp))
    conn.commit(); conn.close()

print('✅ 全関数定義完了（カレンダーベース・騎手・調教師対応）')


✅ 全関数定義完了（カレンダーベース・騎手・調教師対応）


## セル5: 特徴量エンジン

In [16]:
# ── src/features/engine からインポート ─────────────────────────────
import sys as _sys_eng
if BASE_DIR not in _sys_eng.path:
    _sys_eng.path.insert(0, BASE_DIR)

from src.features.engine import (
    init_engine,
    calc_all, calc_pace_distribution, calc_performance_index,
    calc_features_for_xgb, calc_chaos_score, auto_comment,
    f_recent, f_pace, f_dist_v2, f_blood, f_jockey, f_trainer, f_weight, f_post, f_bias,
    analyze_career, apply_career_flags,
    dz, dist_zone_label, get_race_class_rank,
    SIRE_DB, DEF_SIRE, CLASS_RANK, CLASS_RANK_NAME, PACE_STYLE_SCORE,
)

# エンジン初期化（weights/calibrator/pace_model を BASE_DIR/data/ から読み込み）
init_engine(BASE_DIR)


✅ 展開モデル読み込み完了
✅ キャリブレーター読み込み完了
⚠ キャリブレーターなし → 補正なし
⚠ 展開モデルなし → ルールベースで代替
✅ 特徴量エンジン v5 定義完了
✅ 展開モデル組み込み済み


## セル6: 買い目・DB定義

In [17]:
import sqlite3

DB_PATH = f'{BASE_DIR}/data/keiba.db'

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.executescript('''
        CREATE TABLE IF NOT EXISTS races (
            id TEXT PRIMARY KEY, date TEXT, racecourse TEXT,
            race_name TEXT, distance INTEGER, surface TEXT,
            condition TEXT, num_horses INTEGER, raw_json TEXT);
        CREATE TABLE IF NOT EXISTS results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            race_id TEXT, place INTEGER, horse_num INTEGER,
            horse_name TEXT, running_style TEXT,
            agari3f REAL, tansho_payout INTEGER, fukusho_payout INTEGER);
        CREATE TABLE IF NOT EXISTS bets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date TEXT, race_id TEXT, bet_type TEXT,
            horse_num INTEGER, horse_name TEXT,
            odds_est REAL, amount INTEGER,
            is_hit INTEGER DEFAULT -1, payout INTEGER DEFAULT 0,
            horse_num2 INTEGER DEFAULT 0);
        CREATE TABLE IF NOT EXISTS bet_simulation (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date TEXT, race_id TEXT, racecourse TEXT,
            race_num INTEGER, bet_type TEXT,
            horse_num TEXT, horse_name TEXT,
            odds_est REAL, ai_prob REAL, ev REAL,
            num_horses INTEGER, chaos REAL,
            is_tanzen INTEGER, is_2kyou INTEGER, is_konsen INTEGER,
            pop_rank INTEGER, score_gap REAL,
            is_hit INTEGER DEFAULT -1, payout REAL DEFAULT 0
        );
    ''')
    conn.commit(); conn.close()

def save_race_db(race):
    conn = sqlite3.connect(DB_PATH)
    conn.execute('INSERT OR REPLACE INTO races VALUES (?,?,?,?,?,?,?,?,?)',
        (race['id'], race['date'], race['racecourse'], race['race_name'],
         race['distance'], race['surface'], race.get('condition','良'),
         race['num_horses'], json.dumps(race, ensure_ascii=False)))
    conn.commit(); conn.close()

def save_bets_db(date_str, race_id, bets):
    conn = sqlite3.connect(DB_PATH)
    for b in bets:
        # 三連複の複数点処理
        if b['type'] == '三連複' and 'tickets' in b:
            for t in b['tickets']:
                existing = conn.execute(
                    'SELECT id FROM bets WHERE race_id=? AND bet_type=? AND horse_num=? AND horse_num2=?',
                    (race_id, '三連複', t[0], t[1])).fetchone()
                if existing: continue
                conn.execute(
                    'INSERT INTO bets (date,race_id,bet_type,horse_num,horse_name,odds_est,amount,horse_num2) VALUES (?,?,?,?,?,?,?,?)',
                    (date_str, race_id, '三連複', t[0], b.get('horse_name',''), b.get('odds_est',0), 100, t[1]))
            continue
        existing = conn.execute(
            'SELECT id FROM bets WHERE race_id=? AND bet_type=? AND horse_num=?',
            (race_id, b['type'], b['nums'][0])).fetchone()
        if existing: continue
        horse_num2 = b['nums'][1] if len(b['nums']) > 1 else 0
        conn.execute(
            'INSERT INTO bets (date,race_id,bet_type,horse_num,horse_name,odds_est,amount,horse_num2) VALUES (?,?,?,?,?,?,?,?)',
            (date_str, race_id, b['type'], b['nums'][0],
             b.get('horse_name',''), b.get('odds_est',0), b['amount'], horse_num2))
    conn.commit(); conn.close()

def classify_race(race, scored):
    if len(scored) < 3: return 'E', {}
    by_odds = sorted(scored, key=lambda h: h.get('win_odds') or 99)
    num_horses = race.get('num_horses', len(scored))
    o1 = by_odds[0].get('win_odds') or 99
    o2 = by_odds[1].get('win_odds') or 99
    o3 = by_odds[2].get('win_odds') or 99
    score_gap = scored[0]['total'] - scored[1]['total'] if len(scored)>1 else 0
    top5_odds = [h.get('win_odds') or 99 for h in by_odds[:5]]
    import statistics as _st
    odds_cv = _st.stdev(top5_odds) / (_st.mean(top5_odds)+0.001)
    info = {
        'o1':o1,'o2':o2,'o3':o3,
        'num_horses':num_horses,'score_gap':score_gap,'odds_cv':odds_cv,
        'top3':by_odds[:3],'top5':by_odds[:5],
    }
    if o1 <= 1.8 and o2/o1 >= 2.0: return 'A', info
    if score_gap >= 0.5 and 5.0 <= scored[0].get('win_odds',0) <= 15.0: return 'B', info
    if o3/o1 < 2.5 and o1 >= 2.0: return 'C', info
    if odds_cv > 1.0 and num_horses >= 14: return 'D', info
    return 'E', info

def calc_ev(win_prob, odds):
    return round(odds * win_prob, 3)

def calc_kelly(win_prob, odds, bankroll, max_ratio=0.05):
    """
    ケリー基準による最適ベット額
    max_ratio: バンクロールの最大5%まで
    実用上は1/4ケリーで保守的に運用
    """
    if odds <= 1.0 or win_prob <= 0: return 0
    b = odds - 1.0
    kelly_f = (win_prob * b - (1 - win_prob)) / b
    kelly_f = kelly_f * 0.25  # 1/4ケリー（保守的）
    kelly_f = max(0, min(kelly_f, max_ratio))
    amount = int(bankroll * kelly_f / 100) * 100  # 100円単位
    return max(0, amount)

def harville_pair_prob(pi, pj):
    """P(i と j が両方top2) — ワイド・馬連の確率推定（Harville公式）"""
    di = max(1e-9, 1.0 - pi); dj = max(1e-9, 1.0 - pj)
    return min(1.0, pi*pj/di + pj*pi/dj)

def harville_trio_prob(pi, pj, pk):
    """P(i,j,k がすべてtop3) — 三連複の確率推定（Harville公式・全6順列合算）"""
    ps=[pi,pj,pk]; perms=[(0,1,2),(0,2,1),(1,0,2),(1,2,0),(2,0,1),(2,1,0)]; t=0.0
    for a,b,c in perms:
        pa,pb,pc=ps[a],ps[b],ps[c]; da=max(1e-9,1-pa); dab=max(1e-9,1-pa-pb)
        t+=pa*(pb/da)*(pc/dab)
    return min(1.0, t)

# 券種選択モデル読み込み
import pickle as _pkl2
_BET_SELECTOR = None
_BET_SELECTOR_LE = None
_bs_path = f'{BASE_DIR}/data/bet_selector_model.pkl'
_le_path = f'{BASE_DIR}/data/bet_selector_le.pkl'
if os.path.exists(_bs_path) and os.path.exists(_le_path):
    with open(_bs_path,'rb') as _f: _BET_SELECTOR = _pkl2.load(_f)
    with open(_le_path,'rb') as _f: _BET_SELECTOR_LE = _pkl2.load(_f)
    print('✅ 券種選択モデル読み込み完了')
else:
    print('⚠ 券種選択モデルなし → EVルールで代替')
def make_bets(c):
    """EV x 適性係数スコアで最適券種を選択（v5）"""
    import statistics as _st
    race=c['race']; scored=c['scored']; top1=c['top1']
    nh=race.get('num_horses',16); sg=c.get('score_gap',0); ch=c.get('chaos_score',0)
    def go(i,k,d): return scored[i].get(k,d) if len(scored)>i else d
    o1=go(0,'win_odds',10.0) or 10.0; o2=go(1,'win_odds',20.0) or 20.0; o3=go(2,'win_odds',30.0) or 30.0
    p1=go(0,'pn',0.10); p2=go(1,'pn',0.08); p3=go(2,'pn',0.06)
    f1=go(0,'top3_prob',min(0.80,p1*3)); f2=go(1,'top3_prob',min(0.80,p2*3)); f3=go(2,'top3_prob',min(0.80,p3*3))
    itz=(o1<=2.5); i2k=(o2/o1<=1.8 and o3/o2>=1.8); ikn=(ch>=0.55)
    ish=(nh<=8); ita=(nh>=14); imd=(3.0<=o1<=12.0); icl=(sg>=0.5 and o1>=3.0)
    fp=f1; fo=max(1.2,o1*0.30); fev=calc_ev(fp,fo); fa=max(FUKU_AMT,min(calc_kelly(fp,fo,BANKROLL),FUKU_AMT*3))
    tev=calc_ev(p1,o1); ta=max(TAN_AMT,min(calc_kelly(p1,o1,BANKROLL),TAN_AMT*3))
    pair12=harville_pair_prob(p1,p2)
    wp=pair12; wo=max(1.8,(o1*o2)**0.5*0.65); wev=calc_ev(wp,wo); wa=max(WIDE_AMT,min(calc_kelly(wp,wo,BANKROLL),WIDE_AMT*3))
    rp=pair12; ro=max(3.0,(o1*o2)**0.5*1.10); rev=calc_ev(rp,ro); ra=max(REN_AMT,min(calc_kelly(rp,ro,BANKROLL),REN_AMT*3))
    t2p=min(p1*p2/max(1e-9,1-p1),0.50); t2o=max(3.0,(o1*o2)**0.5*1.3); t2ev=calc_ev(t2p,t2o); t2a=max(TAN2_AMT,min(calc_kelly(t2p,t2o,BANKROLL),TAN2_AMT*3))
    sp=harville_trio_prob(p1,p2,p3); so=round(0.75/sp,1) if sp>0 else 99.0; sev=calc_ev(sp,so); sa=max(SAN_AMT,min(calc_kelly(sp,so,BANKROLL),SAN_AMT*3))
    def ws(b,*r):
        s=b
        for f,cc in r: s*=f if cc else (2.0-f)
        return s
    fs=ws(fev,(1.1,ikn),(0.95,ish)); ts=ws(tev,(1.2,icl and imd),(0.7,itz),(0.8,o1<3.5))
    ws2=ws(wev,(1.2,itz or i2k),(1.1,ita),(0.85,ikn)); rs=ws(rev,(1.3,i2k and not itz),(1.1,ish),(0.6,itz),(0.8,ita))
    t2s=ws(t2ev,(1.2,icl and ish),(1.1,icl and imd),(0.6,itz),(0.75,ita)); ss=ws(sev,(1.2,ish),(1.1,not ikn and not itz and not ita),(0.7,ikn),(0.8,ita))
    EV=1.00; cds=[]
    # XGBoostで最適券種を予測
    if _BET_SELECTOR is not None:
        import pandas as _pd2
        X_pred = _pd2.DataFrame([[nh, o1, o2, o3,
                                   o2/o1 if o1>0 else 2.0,
                                   o3/o2 if o2>0 else 2.0,
                                   ch,
                                   race.get('distance',1600),
                                   1 if race.get('surface')=='芝' else 0,
                                   race.get('last_3f',0) or 0]],
                        columns=['n','o1','o2','o3','gap12','gap23',
                                 'chaos','distance','surface','last3f'])
        pred_class = _BET_SELECTOR_LE.classes_[_BET_SELECTOR.predict(X_pred)[0]]
        EV = 0.90  # モデル使用時はEV閾値を下げる
    s0n=scored[0]['num'] if scored else top1['num']; s1n=scored[1]['num'] if len(scored)>1 else 0; s2n=scored[2]['num'] if len(scored)>2 else 0
    s0nm=scored[0]['name'] if scored else top1['name']; s1nm=scored[1]['name'] if len(scored)>1 else ''; s2nm=scored[2]['name'] if len(scored)>2 else ''
    if fev>=EV: cds.append(('複勝',fs,fev,{'type':'複勝','mark':'◎','nums':[top1['num']],'horse_name':top1['name'],'odds':o1,'odds_est':round(fo,1),'amount':fa,'ev':fev,'prob':round(fp,4),'pattern':f'複勝:EV{fev:.2f}xw{fs:.2f}'}))
    if tev>=EV and o1>=3.5: cds.append(('単勝',ts,tev,{'type':'単勝','mark':'◎','nums':[top1['num']],'horse_name':top1['name'],'odds':o1,'odds_est':o1,'amount':ta,'ev':tev,'prob':round(p1,4),'pattern':f'単勝:EV{tev:.2f}xw{ts:.2f}'}))
    if wev>=EV and len(scored)>=2: cds.append(('ワイド',ws2,wev,{'type':'ワイド','mark':'◎○','nums':[s0n,s1n],'horse_name':f'{s0nm}-{s1nm}','odds':wo,'odds_est':round(wo,1),'amount':wa,'ev':wev,'prob':round(wp,4),'pattern':f'ワイド:EV{wev:.2f}xw{ws2:.2f}'}))
    if rev>=EV and len(scored)>=2: cds.append(('馬連',rs,rev,{'type':'馬連','mark':'◎○','nums':[s0n,s1n],'horse_name':f'{s0nm}-{s1nm}','odds':ro,'odds_est':round(ro,1),'amount':ra,'ev':rev,'prob':round(rp,4),'pattern':f'馬連:EV{rev:.2f}xw{rs:.2f}'}))
    if t2ev>=EV and len(scored)>=2: cds.append(('馬単',t2s,t2ev,{'type':'馬単','mark':'◎→○','nums':[s0n,s1n],'horse_name':f'{s0nm}→{s1nm}','odds':t2o,'odds_est':round(t2o,1),'amount':t2a,'ev':t2ev,'prob':round(t2p,4),'pattern':f'馬単:EV{t2ev:.2f}xw{t2s:.2f}'}))
    if sev>=EV and len(scored)>=3:
        from itertools import combinations as _comb
        gap12 = scored[0]['total']-scored[1]['total'] if len(scored)>1 else 0
        gap23 = scored[1]['total']-scored[2]['total'] if len(scored)>2 else 0
        n_av  = min(len(scored), 8)
        if gap12>=0.4 and gap23>=0.3:
            aite    = scored[2:5]
            tickets = [[s0n, s1n, h['num']] for h in aite]
            spat    = f'三連複2頭軸流し({len(tickets)}点)'
        elif gap12>=0.4:
            aite    = scored[1:5]
            tickets = [[s0n, h1['num'], h2['num']] for h1,h2 in _comb(aite,2)]
            spat    = f'三連複1頭軸流し({len(tickets)}点)'
        else:
            n_box   = 5 if ikn else 4
            n_box   = min(n_box, n_av)
            tickets = [[h0['num'],h1['num'],h2['num']] for h0,h1,h2 in _comb(scored[:n_box],3)]
            spat    = f'三連複{n_box}頭BOX({len(tickets)}点)'
        tickets   = tickets[:10]
        san_total = len(tickets)*100
        san_nm    = f"{s0nm}-{s1nm}-流し{len(tickets)}点" if '軸' in spat else f"{s0nm}-BOX{len(tickets)}点"
        cds.append(('三連複',ss,sev,{
            'type':'三連複','mark':'◎',
            'nums':tickets[0],'tickets':tickets,
            'horse_name':san_nm,'odds':so,'odds_est':so,
            'amount':san_total,'unit_amount':100,
            'ev':sev,'prob':round(sp,4),
            'pattern':f'{spat}:EV{sev:.2f}xw{ss:.2f}'}))
    if not cds: return [{'type':'複勝','mark':'◎','nums':[top1['num']],'horse_name':top1['name'],'odds':o1,'odds_est':round(fo,1),'amount':FUKU_AMT,'ev':fev,'prob':round(fp,4),'pattern':f'複勝:EV低({fev:.2f})'}]
    cds.sort(key=lambda x:x[1],reverse=True)
    bets=[cds[0][3]]
    for ct,cs,cev,cb in cds[1:]:
        if cev>=1.10 and ct!=cds[0][0]:
            sb=dict(cb); sb['amount']=max(300,cb['amount']//2); sb['pattern']+=':サブ'; bets.append(sb); break
    return bets

def log_bet_simulation(date_str, c):
    """全券種を買った想定で記録（データ蓄積後にROI分析用）"""
    race=c['race']; scored=c['scored']
    rid=race.get('id',''); rc=race.get('racecourse',''); rnum=race.get('race_num',0)
    nh=race.get('num_horses',16); ch=c.get('chaos_score',0); sg=c.get('score_gap',0)
    def go(i,k,d): return scored[i].get(k,d) if len(scored)>i else d
    o1=go(0,'win_odds',10.0) or 10.0; o2=go(1,'win_odds',20.0) or 20.0; o3=go(2,'win_odds',30.0) or 30.0
    p1=go(0,'pn',0.10); p2=go(1,'pn',0.08); p3=go(2,'pn',0.06)
    f1=go(0,'top3_prob',min(0.80,p1*3)); f2=go(1,'top3_prob',min(0.80,p2*3)); f3=go(2,'top3_prob',min(0.80,p3*3))
    itz=1 if o1<=2.5 else 0; i2k=1 if(o2/o1<=1.8 and o3/o2>=1.8) else 0; ikn=1 if ch>=0.55 else 0
    pr=next((i+1 for i,h in enumerate(sorted(scored,key=lambda h:h.get('win_odds') or 99)) if h['name']==scored[0]['name']),99)
    s0n=str(scored[0]['num']); s0nm=scored[0]['name']
    rows=[('複勝',s0n,s0nm,max(1.1,o1*0.28),p1,calc_ev(min(0.95,p1*3),max(1.1,o1*0.28))),('単勝',s0n,s0nm,o1,p1,calc_ev(p1,o1))]
    if len(scored)>=2:
        s1n=str(scored[1]['num']); s1nm=scored[1]['name']
        wo=max(1.5,(o1*o2)**0.5*0.45); ro=max(2.0,(o1*o2)**0.5*0.75); t2o=max(3.0,(o1*o2)**0.5*1.3)
        rows+=[('ワイド',f'{s0n}-{s1n}',f'{s0nm}-{s1nm}',wo,min(0.60,p1*p2*6),calc_ev(min(0.60,p1*p2*6),wo)),
               ('馬連',f'{s0n}-{s1n}',f'{s0nm}-{s1nm}',ro,min(0.40,p1*p2*2),calc_ev(min(0.40,p1*p2*2),ro)),
               ('馬単',f'{s0n}->{s1n}',f'{s0nm}->{s1nm}',t2o,min(0.30,p1*p2*1.2),calc_ev(min(0.30,p1*p2*1.2),t2o))]
    if len(scored)>=3:
        s2n=str(scored[2]['num']); s2nm=scored[2]['name']
        ps=min(0.50,p1*p2*p3*6); soo=round(0.75/ps,1) if ps>0 else 99.0
        rows.append(('三連複',f'{s0n}-{scored[1]["num"]}-{s2n}',f'{s0nm}-{scored[1]["name"]}-{s2nm}',soo,ps,calc_ev(ps,soo)))
    conn=sqlite3.connect(DB_PATH)
    for bt,hn,hname,oe,ap,ev in rows:
        conn.execute('INSERT INTO bet_simulation(date,race_id,racecourse,race_num,bet_type,horse_num,horse_name,odds_est,ai_prob,ev,num_horses,chaos,is_tanzen,is_2kyou,is_konsen,pop_rank,score_gap)VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
            (date_str,rid,rc,rnum,bt,hn,hname,round(oe,2),round(ap,4),round(ev,3),nh,round(ch,3),itz,i2k,ikn,pr,round(sg,3)))
    conn.commit(); conn.close()


def ability_first_loose(races, bias_data=None, top_n=6):
    """厳選ロジック（純粋EV判断版・オッズ上限20倍）"""
    EV_THRESHOLD = 1.05
    cands = []
    for race in races:
        scored = calc_all(race, bias_data)
        if len(scored) < 3: continue
        top1 = scored[0]
        odds = top1.get('win_odds') or 99
        if odds < 1.3: continue
        if odds > 20.0: continue  # オッズ上限20倍
        # 未勝利・新馬は推奨しない（精度が低いため・的中率蓄積後に解禁）
        race_class = race.get('class', '')
        if race_class in ('未勝利', '新馬'):
            continue
        gap = top1['total'] - scored[1]['total']
        if gap < 0.01: continue
        win_prob = top1.get('pn', 0)
        # ★ AI勝率フィルター（pn < 0.06 は信頼性不足でスキップ）
        if win_prob < 0.06:
            continue
        ev_fuku  = calc_ev(min(1.0, win_prob * 3), odds * 0.28)
        ev_tan   = calc_ev(win_prob, odds)
        ev_max   = max(ev_fuku, ev_tan)
        if ev_max < EV_THRESHOLD: continue
        by_odds  = sorted(scored, key=lambda h: h.get('win_odds') or 99)
        pop_rank = next((i+1 for i,h in enumerate(by_odds) if h['name']==top1['name']), 99)
        cands.append({
            'race':race,'scored':scored,'top1':top1,
            'odds':odds,'popularity_rank':pop_rank,
            'score_gap':gap,
            'priority': ev_max * gap,
            'ev_fuku':ev_fuku,'ev_tan':ev_tan,'ev_max':ev_max,
            'chaos_score':calc_chaos_score(race, scored),
        })
    cands.sort(key=lambda x: x['priority'], reverse=True)
    selected = cands[:top_n]
    selected.sort(key=lambda x: (
        VENUE_ORDER.get(x['race']['racecourse'], 99),
        x['race']['race_num']
    ))
    return selected

def to_app_json(selected, races_all, bias_data, jst_now, day_type='friday'):
    all_venues = sorted({r['racecourse'] for r in races_all},
                        key=lambda v: VENUE_ORDER.get(v,99))
    total_inv  = sum(sum(b['amount'] for b in make_bets(c)) for c in selected)
    races_by_venue = {}
    selected_ids = {c['race']['id'] for c in selected}

    # 厳選レース
    for c in selected:
        race=c['race']; top1=c['top1']; scored=c['scored']
        bets=make_bets(c); rc=race['racecourse']
        if rc not in races_by_venue: races_by_venue[rc]=[]
        bet_list=[]
        for b in bets:
            tag='fuku' if b['type']=='複勝' else 'tan' if b['type']=='単勝' else 'wide'
            bet_list.append({'tag':tag,'label':b['type'],
                'horse':f'#{b["nums"][0]} {b["horse_name"]}',
                'est':f'推定{b["odds_est"]:.1f}倍' if b['type']=='複勝' else f'{b["odds"]:.1f}倍',
                'amt':f'¥{b["amount"]}'})
        by_odds=sorted(scored,key=lambda h: h.get('win_odds') or 99)
        pop_rank=next((i+1 for i,h in enumerate(by_odds) if h['name']==top1['name']),99)
        conf=min(99,max(50,int(60+(pop_rank-2)*4+c['score_gap']*20)))
        _s = sum(h.get('pn',0) for h in scored) or 1
        _ev_list = []
        for _h in scored:
            _p1  = _h.get('pn', 0.1)
            _o1  = _h.get('win_odds', 10) or 10
            _fp  = min(0.88, _p1 * 3.2)
            _fo  = max(1.2, _o1 * 0.32)
            _ev_list.append(round(_fp * _fo, 3))
        _ev_ranks = {scored[_i]['num']: _r+1 for _r,_i in enumerate(
            sorted(range(len(scored)), key=lambda x: _ev_list[x], reverse=True))}
        _rl_ranks = {scored[_i]['num']: _r+1 for _r,_i in enumerate(
            sorted(range(len(scored)), key=lambda x: scored[x]['total'], reverse=True))}
        _cl_scores_list = [scored[_i].get('scores',{}).get('jockey',5)
                          + scored[_i].get('scores',{}).get('blood',5)
                          + scored[_i].get('scores',{}).get('distance',5)
                          for _i in range(len(scored))]
        _cl_ranks = {scored[_i]['num']: _r+1 for _r,_i in enumerate(
            sorted(range(len(scored)), key=lambda x: _cl_scores_list[x], reverse=True))}
        horses_list=[{
            'n':h['num'],'name':h['name'],
            'odds':h.get('win_odds',0) or 0,
            'score':round(h['total'],1),
            'pop':next((i+1 for i,x in enumerate(by_odds) if x['name']==h['name']),99),
            'style':h.get('running_style','差し'),
            'tan_pct':round(min(60, h.get('pn',0)*100), 1),
            'ren_pct':round(min(80, h.get('pn',0)*2.0*100), 1),
            'fuku_pct':round(min(88, h.get('pn',0)*3.2*100), 1),
            'ev_rank':_ev_ranks.get(h['num'], 99),
            'rl_rank':_rl_ranks.get(h['num'], 99),
            'cl_rank':_cl_ranks.get(h['num'], 99),
            'mark':('推' if h['num']==top1['num'] else
                    '高' if round(min(88,h.get('pn',0)*3.2*100),1)>=38 else
                    '穴' if (_rl_ranks.get(h['num'],99)<=4 and (h.get('win_odds',0) or 0)>=10) else ''),
        } for h in scored]
        horses_list.sort(key=lambda x: x['n'])
        races_by_venue[rc].append({
            'r':race['race_num'],'name':race['race_name'],
            'dist':f'{race["distance"]}m{race["surface"]}',
            'rec':True,'conf':conf,
            'honmei':{'n':top1['num'],'name':top1['name'],
                      'odds':top1.get('win_odds',0) or 0,
                      'score':top1['total'],'style':top1.get('running_style','差し')},
            'horses':horses_list,'bets':bet_list,
            'chaos_label':('A' if c.get('chaos_score',0)>=0.65 else 'B' if c.get('chaos_score',0)>=0.45 else 'C' if c.get('chaos_score',0)>=0.25 else 'D'),
            'cmt':auto_comment(c,bias_data)})

    # 非厳選レース（全レース買い目・的中指数つき）
    for race in sorted(races_all, key=lambda r:(VENUE_ORDER.get(r['racecourse'],99),r['race_num'])):
        if race['id'] in selected_ids: continue
        rc=race['racecourse']
        if rc not in races_by_venue: races_by_venue[rc]=[]
        scored=calc_all(race,bias_data)
        if not scored: continue
        top1=scored[0]
        odds=top1.get('win_odds') or 99
        gap=top1['total']-scored[1]['total'] if len(scored)>1 else 0
        by_odds=sorted(scored,key=lambda h: h.get('win_odds') or 99)
        pop_rank=next((i+1 for i,h in enumerate(by_odds) if h['name']==top1['name']),99)
        conf=min(99,max(1,int(60+(pop_rank-2)*4+gap*20)))
        win_prob=top1.get('pn',0)
        ev_fuku=calc_ev(min(1.0,win_prob*3), odds*0.28)
        ev_tan =calc_ev(win_prob, odds)
        c_ref={'race':race,'scored':scored,'top1':top1,'odds':odds,
               'popularity_rank':pop_rank,'score_gap':gap,
               'ev_fuku':ev_fuku,'ev_tan':ev_tan,'ev_max':max(ev_fuku,ev_tan)}
        bets=make_bets(c_ref)
        bet_list=[]
        for b in bets:
            tag='fuku' if b['type']=='複勝' else 'tan' if b['type']=='単勝' else 'wide'
            bet_list.append({'tag':tag,'label':b['type'],
                'horse':f'#{b["nums"][0]} {b["horse_name"]}',
                'est':f'推定{b["odds_est"]:.1f}倍' if b['type']=='複勝' else f'{b["odds"]:.1f}倍',
                'amt':f'¥{b["amount"]}'})
        _s = sum(h.get('pn',0) for h in scored) or 1
        _ev_list = []
        for _h in scored:
            _p1  = _h.get('pn', 0.1)
            _o1  = _h.get('win_odds', 10) or 10
            _fp  = min(0.88, _p1 * 3.2)
            _fo  = max(1.2, _o1 * 0.32)
            _ev_list.append(round(_fp * _fo, 3))
        _ev_ranks = {scored[_i]['num']: _r+1 for _r,_i in enumerate(
            sorted(range(len(scored)), key=lambda x: _ev_list[x], reverse=True))}
        _rl_ranks = {scored[_i]['num']: _r+1 for _r,_i in enumerate(
            sorted(range(len(scored)), key=lambda x: scored[x]['total'], reverse=True))}
        _cl_scores_list = [scored[_i].get('scores',{}).get('jockey',5)
                          + scored[_i].get('scores',{}).get('blood',5)
                          + scored[_i].get('scores',{}).get('distance',5)
                          for _i in range(len(scored))]
        _cl_ranks = {scored[_i]['num']: _r+1 for _r,_i in enumerate(
            sorted(range(len(scored)), key=lambda x: _cl_scores_list[x], reverse=True))}
        horses_list=[{
            'n':h['num'],'name':h['name'],
            'odds':h.get('win_odds',0) or 0,
            'score':round(h['total'],1),
            'pop':next((i+1 for i,x in enumerate(by_odds) if x['name']==h['name']),99),
            'style':h.get('running_style','差し'),
            'tan_pct':round(min(60, h.get('pn',0)*100), 1),
            'ren_pct':round(min(80, h.get('pn',0)*2.0*100), 1),
            'fuku_pct':round(min(88, h.get('pn',0)*3.2*100), 1),
            'ev_rank':_ev_ranks.get(h['num'], 99),
            'rl_rank':_rl_ranks.get(h['num'], 99),
            'cl_rank':_cl_ranks.get(h['num'], 99),
            'mark':('推' if h['num']==top1['num'] else
                    '高' if round(min(88,h.get('pn',0)*3.2*100),1)>=38 else
                    '穴' if (_rl_ranks.get(h['num'],99)<=4 and (h.get('win_odds',0) or 0)>=10) else ''),
        } for h in scored]
        horses_list.sort(key=lambda x: x['n'])
        races_by_venue[rc].append({
            'r':race['race_num'],'name':race['race_name'],
            'dist':f'{race["distance"]}m{race["surface"]}',
            'rec':False,'conf':conf,
            'honmei':{'n':top1['num'],'name':top1['name'],
                      'odds':top1.get('win_odds',0) or 0,
                      'score':top1['total'],'style':top1.get('running_style','差し')},
            'horses':horses_list,'bets':bet_list,
            'cmt':auto_comment(c_ref,bias_data)})

    bias_txt='内外:フラット ペース:±0 時計:±0'; bias_tag='フラット'
    if bias_data:
        bias_txt=bias_data.get('summary',bias_txt)
        spd=bias_data.get('track_speed',0)
        bias_tag='時計速め' if spd>0.3 else '時計遅め' if spd<-0.3 else 'フラット'
    jday=['月','火','水','木','金','土','日'][jst_now.weekday()]
    return {
        'generated_at':jst_now.isoformat(),
        'date':f'{jst_now.month}月{jst_now.day}日({jday})',
        'type':day_type,'venues':all_venues,
        'bias':{'txt':bias_txt,'tag':bias_tag},
        'stats':{'invest':total_inv,'rec':len(selected),'roi':150},
        'races':races_by_venue
    }

init_db()
print('✅ 買い目・DB・ability_first_loose・to_app_json 定義完了')
print('   EV閾値: 1.05 / サブEV閾値: 1.10 / pnフィルター: 0.06 / オッズ上限: 30倍')
print('   対象券種: 複勝・単勝・ワイド・馬連・馬単・三連複（EV×条件スコア選択）')
print('init_db (with bet_simulation) defined')
print('make_bets (v2 weighted score) defined')
print('log_bet_simulation defined')
# ── src/betting/ からインポート ────────────────────────────────────
import sys as _sys
if BASE_DIR not in _sys.path:
    _sys.path.insert(0, BASE_DIR)

from src.betting.make_bets import (
    init_betting,
    calc_ev, calc_kelly,
    make_bets,
    log_bet_simulation as _log_bet_simulation,
)
from src.betting.ev_filter import ability_first_loose
from src.betting.app_json import to_app_json

def log_bet_simulation(date_str, c):
    return _log_bet_simulation(date_str, c, db_path=DB_PATH)

init_betting(BASE_DIR,
             bankroll=BANKROLL,
             fuku_amt=FUKU_AMT, tan_amt=TAN_AMT, wide_amt=WIDE_AMT,
             ren_amt=REN_AMT,   tan2_amt=TAN2_AMT, san_amt=SAN_AMT)

print('✅ src/betting/ から読み込み完了')


✅ 券種選択モデル読み込み完了
✅ 買い目・DB・ability_first_loose・to_app_json 定義完了
   EV閾値: 1.05 / サブEV閾値: 1.10 / pnフィルター: 0.06 / オッズ上限: 30倍
   対象券種: 複勝・単勝・ワイド・馬連・馬単・三連複（EV×条件スコア選択）
init_db (with bet_simulation) defined
make_bets (v2 weighted score) defined
log_bet_simulation defined


## セル7: DB修正（初回のみ）

In [18]:
# ① betsテーブルにhorse_num2列がなければ追加
import sqlite3
conn = sqlite3.connect(DB_PATH)
try:
    conn.execute('ALTER TABLE bets ADD COLUMN horse_num2 INTEGER DEFAULT 0')
    conn.commit()
    print('✅ horse_num2列を追加しました')
except Exception as e:
    print(f'✅ horse_num2列: {e}')
conn.close()
init_db()
print('✅ DB準備完了')


✅ horse_num2列: duplicate column name: horse_num2
✅ DB準備完了


## セル8: ★ 土曜レース取得

In [19]:
sess = requests.Session()
sess.headers.update(HEADERS)
sess.get(f'{JRA_BASE}/keiba/thisweek/', timeout=15)

jst_now  = get_jst_now()
from datetime import timedelta

# 金曜v5: 土曜未明実行 → 今日が土曜
sat_date = jst_now.strftime('%Y%m%d')
print(f'土曜取得対象日: {sat_date}')

races_sat = fetch_races_on_date(sess, sat_date, HIST_PATH)
print(f'取得完了: {len(races_sat)}レース')
for r in races_sat:
    print(f'  {r["racecourse"]}R{r["race_num"]:02d} {r["race_name"]} {r["distance"]}m{r["surface"]}')

races = races_sat

# バイアスは保存済みデータを使用（土曜結果はまだないため）
avg_bias = None
bf = f'{BASE_DIR}/data/track_bias_latest.json'
if os.path.exists(bf):
    with open(bf) as _bf: avg_bias = json.load(_bf)
    print(f'バイアス読み込み: {avg_bias.get("date","")} {avg_bias.get("summary","フラット")}')
else:
    print('バイアス: なし（フラット想定）')
bias = avg_bias


土曜取得対象日: 20260516
📡 20260516 レース取得中...
  📅 東京 → pw01dde010520260207
  📅 阪神 → pw01dde010920260302
  📅 京都 → pw01dde010820260307
  📅 新潟 → pw01dde010420260105

🏟 東京  suffix探索... ✅ 73
  R01: 3歳未勝利 400m ダート 16頭
  R02: 3歳未勝利 600m ダート 16頭
  R03: 3歳未勝利 300m ダート 16頭
  R04: 3歳未勝利 0m 芝 13頭
  R05: 3歳未勝利 400m 芝 18頭
  R06: 3歳1勝クラス 400m 芝 8頭
  R07: 3歳1勝クラス 400m ダート 16頭
  R08: 4歳以上1勝クラス 600m ダート 16頭
  R09: 調布特別 800m 芝 9頭
  R10: 立川特別 400m ダート 12頭
  R11: 六社ステークス 400m 芝 15頭
  R12: 4歳以上1勝クラス 600m ダート 14頭

🏟 阪神  suffix探索... ❌

🏟 京都  suffix探索... ✅ 82
  R01: 3歳未勝利 400m ダート 16頭
  R02: 3歳未勝利 800m ダート 11頭
  R03: 3歳未勝利 400m 芝 18頭
  R04: 3歳未勝利 400m 芝 10頭
  R05: 3歳未勝利 800m 芝 14頭
  R06: 4歳以上1勝クラス 200m ダート 15頭
  R07: 4歳以上1勝クラス 600m 芝 11頭
  R09: あずさ賞 0m 芝 10頭
  R10: 上賀茂ステークス 900m ダート 15頭
  R11: 鞍馬ステークス 200m 芝 16頭
  R12: 4歳以上2勝クラス 200m ダート 15頭

🏟 新潟  suffix探索... ✅ 12
  R01: 3歳未勝利 800m ダート 15頭
  R02: 3歳未勝利 200m ダート 15頭
  R03: 3歳未勝利 600m 芝 16頭
  R04: 3歳未勝利 800m ダート 11頭
  R05: 3歳未勝利 800m 芝 16頭
  R06: 3歳未勝利 2000m 芝 16頭
  R

## セル9: ★ 土曜予想生成・保存

In [20]:
# 土曜予想生成（バイアス反映済み）
selected_sat = ability_first_loose(races_sat, avg_bias, top_n=TOP_N_RACES)
total_inv_sat = 0
lines_sat = [f'🏇 KEIBA-AI 土曜予想 {jst_now.strftime("%m/%d(%a)")}',
             f'バイアス: {avg_bias["summary"] if avg_bias else "なし"}',
             '='*40, '']

if selected_sat:
    lines_sat.append(f'【★厳選 {len(selected_sat)}レース】')
    for i, c in enumerate(selected_sat, 1):
        race=c['race']; top1=c['top1']; scored=c['scored']
        bets=make_bets(c)
        _ev_list2=[round(scored[_j].get('pn',0)*max(1.2,scored[_j].get('win_odds',10)*0.32),3) for _j in range(len(scored))]
        _ev_ranks={scored[_j]['num']:_r+1 for _r,_j in enumerate(sorted(range(len(scored)),key=lambda x:_ev_list2[x],reverse=True))}
        save_race_db(race)
        save_bets_db(race['date'], race['id'], bets)
        log_bet_simulation(race['date'], c)
        invest=sum(b['amount'] for b in bets); total_inv_sat+=invest
        bet_parts=[]
        for b in bets:
            ev_str=f" EV{b.get('ev',0):.2f}" if b.get('ev',0)>0 else ''
            if b['type']=='複勝':
                bet_parts.append(f"{b['mark']}複勝{b['nums'][0]} 推定{b['odds_est']}倍 ¥{b['amount']}{ev_str}")
            elif b['type']=='単勝':
                bet_parts.append(f"{b['mark']}単勝{b['nums'][0]} {b['odds']:.1f}倍 ¥{b['amount']}{ev_str}")
            elif b['type']=='ワイド':
                bet_parts.append(f"{b['mark']}W{b['nums'][0]}-{b['nums'][1]} 推定{b['odds_est']}倍 ¥{b['amount']}")
            elif b['type']=='馬連':
                bet_parts.append(f"{b['mark']}馬連{b['nums'][0]}-{b['nums'][1]} 推定{b['odds_est']:.1f}倍 ¥{b['amount']}{ev_str}")
            elif b['type']=='馬単':
                bet_parts.append(f"{b['mark']}馬単{b['nums'][0]}→{b['nums'][1]} 推定{b['odds_est']:.1f}倍 ¥{b['amount']}{ev_str}")
            elif b['type']=='三連複':
                _tix=b.get('tickets',[b['nums']])
                _ts=' / '.join([f"{t[0]}-{t[1]}-{t[2]}" for t in _tix[:3]])
                if len(_tix)>3: _ts+=f'...他{len(_tix)-3}点'
                bet_parts.append(f"◎三連複 {_ts} ¥100×{len(_tix)}点=¥{b['amount']:,} EV{b.get('ev',0):.2f}")
        lines_sat.append(f'★[{i}] {race["racecourse"]}R{race["race_num"]:02d} {race["race_name"]} {race["distance"]}m{race["surface"]}')
        lines_sat.append(f'  ◎#{top1["num"]} {top1["name"]} {top1.get("win_odds",0):.1f}倍 (能力1位/人気{c["popularity_rank"]}位) スコア:{top1["total"]:.1f}')
        lines_sat.append(f'  {"  ".join(bet_parts)}')
        lines_sat.append(auto_comment(c, avg_bias)); lines_sat.append('')
else:
    lines_sat.append('【厳選レースなし】')

# 全レース参考
lines_sat.append('='*40)
lines_sat.append('【全レース参考予想】')
from collections import defaultdict
by_venue_sat = defaultdict(list)
selected_ids_sat = {c['race']['id'] for c in selected_sat}
for race in sorted(races_sat, key=lambda r:(VENUE_ORDER.get(r['racecourse'],99),r['race_num'])):
    scored = calc_all(race, avg_bias)
    if not scored: continue
    top1=scored[0]
    by_odds_r=sorted(scored,key=lambda h: h.get('win_odds') or 99)
    pop1=next((i+1 for i,h in enumerate(by_odds_r) if h['name']==top1['name']),99)
    by_venue_sat[race['racecourse']].append((race,top1,pop1,race['id'] in selected_ids_sat))

for venue in sorted(by_venue_sat.keys(),key=lambda v:VENUE_ORDER.get(v,99)):
    lines_sat.append(f'\n■ {venue}')
    for race,t1,pop1,is_sel in by_venue_sat[venue]:
        mark='★' if is_sel else '　'
        lines_sat.append(f'{mark}R{race["race_num"]:02d} {race["race_name"][:8]} {race["distance"]}m{race["surface"]} ◎#{t1["num"]}{t1["name"][:5]}({t1.get("win_odds",0):.1f}倍/{pop1}番人気)')

full_sat = '\n'.join(lines_sat)
print(full_sat)

# アプリJSON保存
app_data_sat = to_app_json(selected_sat, races_sat, avg_bias, jst_now, day_type='saturday')
json_path = f'{BASE_DIR}/app/prediction_latest.json'
os.makedirs(f'{BASE_DIR}/app', exist_ok=True)
# 土曜の結果も含める
if os.path.exists(json_path):
    with open(json_path) as f: existing = json.load(f)
    app_data_sat['saturday_result'] = existing.get('saturday_result', {})
with open(json_path,'w',encoding='utf-8') as f:
    json.dump(app_data_sat,f,ensure_ascii=False,indent=2)
print(f'\n✅ アプリJSON保存: {json_path}')
log_sat = f'{BASE_DIR}/logs/saturday_{jst_now.strftime("%Y%m%d_%H%M")}.txt'
with open(log_sat,'w') as f: f.write(full_sat)
print(f'✅ ログ保存: {log_sat}')


🏇 KEIBA-AI 土曜予想 05/16(Sat)
バイアス: フラット

【★厳選 5レース】
★[1] 東京R09 調布特別 800m芝
  ◎#8 ラビットアイ 13.7倍 (能力1位/人気6位) スコア:6.7
  ◎単勝8 13.7倍 ¥900 EV2.42  ◎複勝8 推定4.4倍 ¥750 EV2.33
馬場はフラット。ミドル想定(99%)で差し有利な展開。やや拮抗したメンバー構成。◎ラビットアイ(スコア6.7)が最有力。穴馬(13.7倍)で高配当狙い。

★[2] 京都R07 4歳以上1勝クラス 600m芝
  ◎#7 グロリアスグレイス 16.1倍 (能力1位/人気7位) スコア:7.5
  ◎単勝7 16.1倍 ¥900 EV2.54  ◎複勝7 推定5.2倍 ¥750 EV2.44
馬場はフラット。ミドル想定(99%)で差し有利な展開。やや拮抗したメンバー構成。◎グロリアスグレイス(スコア7.5)が最有力。穴馬(16.1倍)で高配当狙い。

★[3] 京都R09 あずさ賞 0m芝
  ◎#8 ワンダフルデイズ 10.6倍 (能力1位/人気7位) スコア:7.3
  ◎単勝8 10.6倍 ¥900 EV1.78  ◎複勝8 推定3.4倍 ¥750 EV1.71
馬場はフラット。ミドル想定(99%)で差し有利な展開。やや拮抗したメンバー構成。◎ワンダフルデイズ(スコア7.3)が最有力。穴馬(10.6倍)で高配当狙い。

★[4] 京都R12 4歳以上2勝クラス 200mダート
  ◎#11 テラステラ 18.9倍 (能力1位/人気9位) スコア:7.3
  ◎単勝11 18.9倍 ¥900 EV1.96  ◎複勝11 推定6.0倍 ¥750 EV1.88
馬場はフラット。ミドル想定(99%)で追込有利な展開。混戦模様で波乱含み。◎テラステラ(スコア7.3)が最有力。穴馬(18.9倍)で高配当狙い。

★[5] 新潟R07 4歳以上1勝クラス 200mダート
  ◎#12 オースミライト 11.0倍 (能力1位/人気7位) スコア:7.3
  ◎単勝12 11.0倍 ¥500 EV1.21  ◎複勝12 推定3.5倍 ¥750 EV1.16
馬場はフラット。ミドル想定(99%)で先行有利な展開。混戦模様で波乱含み。◎オースミライト(スコア7.3)が